# 面试问题：Prefix Cache 感知的 LLM 副本路由怎样同时考虑缓存命中与排队延迟？

        ## 可直接复述的回答主线

        1. 只按最短队列路由会把带有大公共前缀的请求送到无缓存副本，重复 prefill 并拉高 TTFT。
2. 路由器要从每个副本读取队列等待、prefill 吞吐和已缓存前缀 Token 数，估计该请求的真实首 Token 成本。
3. 候选成本可以写成 queue_ms + uncached_prompt_tokens / prefill_tps，并保留每项分数用于解释。
4. 同一前缀的缓存命中只是软亲和性；缓存副本过载时，成本模型应允许选择轻载但 miss 的副本。
5. 评测应在同一批业务请求上逐条比较路由、命中 Token、TTFT 与 SLO，而不是只统计 cache hit。
6. 生产还需缓存目录一致性、前缀哈希、KV 版本、热点复制、失效传播、路由陈旧度和尾延迟监控。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是三个推理副本和七条企业请求，分别复用客服政策、法律模板、代码规范三类长 system prompt。每条请求包含前缀长度、私有问题长度和 TTFT SLO；副本缓存目录及队列延迟是确定性的离线快照，不能外推为线上吞吐。

In [1]:
import math  # 使用基础算术估算未缓存 prefill 的毫秒成本。
replicas = [{"id": "replica-a", "queue_ms": 75.0, "prefill_tps": 8000.0, "cache": {"legal-v3": 920, "support-v5": 0, "code-v2": 0}}, {"id": "replica-b", "queue_ms": 12.0, "prefill_tps": 8000.0, "cache": {"legal-v3": 0, "support-v5": 780, "code-v2": 0}}, {"id": "replica-c", "queue_ms": 48.0, "prefill_tps": 8000.0, "cache": {"legal-v3": 0, "support-v5": 0, "code-v2": 1040}}]  # 定义三个副本的排队延迟、prefill速度和前缀缓存目录。
requests = [{"id": "route-01", "tenant": "legal", "prefix": "legal-v3", "prefix_tokens": 920, "unique_tokens": 80, "slo_ms": 100.0}, {"id": "route-02", "tenant": "support", "prefix": "support-v5", "prefix_tokens": 780, "unique_tokens": 55, "slo_ms": 90.0}, {"id": "route-03", "tenant": "developer", "prefix": "code-v2", "prefix_tokens": 1040, "unique_tokens": 120, "slo_ms": 140.0}, {"id": "route-04", "tenant": "legal", "prefix": "legal-v3", "prefix_tokens": 920, "unique_tokens": 40, "slo_ms": 95.0}, {"id": "route-05", "tenant": "support", "prefix": "support-v5", "prefix_tokens": 780, "unique_tokens": 160, "slo_ms": 110.0}, {"id": "route-06", "tenant": "developer", "prefix": "code-v2", "prefix_tokens": 1040, "unique_tokens": 45, "slo_ms": 125.0}, {"id": "route-07", "tenant": "legal", "prefix": "legal-v3", "prefix_tokens": 920, "unique_tokens": 210, "slo_ms": 125.0}]  # 定义七条带长共享前缀和私有问题的请求。
print("教学实验输入：Prefix Cache 路由快照")  # 标记下方数据是离线可复现样例。
print("副本        queue_ms  prefill_tps  prefix cache")  # 输出副本状态表头。
for replica in replicas:  # 逐副本展示排队和缓存目录。
    print(f"{replica['id']:<12} {replica['queue_ms']:>8.1f} {replica['prefill_tps']:>12.0f}  {replica['cache']}")  # 输出当前副本的路由输入。
print("请求       tenant     prefix       prefix/unique  SLO")  # 输出请求预览表头。
for request in requests:  # 逐条展示真实业务字段。
    print(f"{request['id']:<10} {request['tenant']:<10} {request['prefix']:<12} {request['prefix_tokens']:>6}/{request['unique_tokens']:<6} {request['slo_ms']:>5.0f}ms")  # 输出当前请求的前缀、私有 Token 和 SLO。

教学实验输入：Prefix Cache 路由快照
副本        queue_ms  prefill_tps  prefix cache
replica-a        75.0         8000  {'legal-v3': 920, 'support-v5': 0, 'code-v2': 0}
replica-b        12.0         8000  {'legal-v3': 0, 'support-v5': 780, 'code-v2': 0}
replica-c        48.0         8000  {'legal-v3': 0, 'support-v5': 0, 'code-v2': 1040}
请求       tenant     prefix       prefix/unique  SLO
route-01   legal      legal-v3        920/80       100ms
route-02   support    support-v5      780/55        90ms
route-03   developer  code-v2        1040/120      140ms
route-04   legal      legal-v3        920/40        95ms
route-05   support    support-v5      780/160      110ms
route-06   developer  code-v2        1040/45       125ms
route-07   legal      legal-v3        920/210      125ms


## 2. Baseline / 基线：始终选择 queue_ms 最小的副本

`replica-b` 当前最空，因此基线把所有请求都送过去。只有客服前缀命中；法律和代码请求会重复计算数百到上千个 Token。

In [2]:
def route_cost(request, replica):  # 计算一个请求落到一个副本后的真实 TTFT 估计。
    cached_tokens = min(replica["cache"].get(request["prefix"], 0), request["prefix_tokens"])  # 读取同版本前缀的可复用 Token 数。
    uncached_tokens = request["prefix_tokens"] + request["unique_tokens"] - cached_tokens  # 计算仍需执行 prefill 的 Token 数。
    prefill_ms = uncached_tokens / replica["prefill_tps"] * 1000.0  # 用副本吞吐把未缓存 Token 转为毫秒。
    ttft_ms = replica["queue_ms"] + prefill_ms  # 合并排队等待与未缓存 prefill 成本。
    return {"replica": replica["id"], "cached_tokens": cached_tokens, "uncached_tokens": uncached_tokens, "queue_ms": replica["queue_ms"], "prefill_ms": prefill_ms, "ttft_ms": ttft_ms}  # 返回可解释的成本分项。
least_queue_replica = min(replicas, key=lambda replica: (replica["queue_ms"], replica["id"]))  # 找到朴素最短队列副本。
baseline_rows = []  # 保存七条请求的基线路由结果。
for request in requests:  # 对同一批请求应用最短队列策略。
    cost = route_cost(request, least_queue_replica)  # 计算被选副本上的真实缓存和 TTFT。
    baseline_rows.append({"id": request["id"], **cost, "slo_met": cost["ttft_ms"] <= request["slo_ms"]})  # 保存路由、延迟和 SLO 结果。
print("Baseline 最短队列路由")  # 标记下表未使用前缀缓存亲和性。
print("请求       replica      cached  uncached  queue_ms  prefill_ms  TTFT_ms  SLO")  # 输出基线结果表头。
for row in baseline_rows:  # 逐请求展示重复 prefill 成本。
    print(f"{row['id']:<10} {row['replica']:<12} {row['cached_tokens']:>6} {row['uncached_tokens']:>9} {row['queue_ms']:>9.1f} {row['prefill_ms']:>11.1f} {row['ttft_ms']:>8.1f} {str(row['slo_met']):>5}")  # 输出当前请求的可解释延迟。

Baseline 最短队列路由
请求       replica      cached  uncached  queue_ms  prefill_ms  TTFT_ms  SLO
route-01   replica-b         0      1000      12.0       125.0    137.0 False
route-02   replica-b       780        55      12.0         6.9     18.9  True
route-03   replica-b         0      1160      12.0       145.0    157.0 False
route-04   replica-b         0       960      12.0       120.0    132.0 False
route-05   replica-b       780       160      12.0        20.0     32.0  True
route-06   replica-b         0      1085      12.0       135.6    147.6 False
route-07   replica-b         0      1130      12.0       141.2    153.2 False


## 3. 底层实现：逐副本计算 queue + uncached prefill

不把“命中”直接当作硬路由规则，而是对每个候选计算同一单位的毫秒成本。这样可以看到缓存节省是否足以抵消额外排队。

In [3]:
def prefix_aware_route(request, replica_snapshot):  # 按总 TTFT 成本选择 Prefix Cache 感知副本。
    candidates = [route_cost(request, replica) for replica in replica_snapshot]  # 计算所有副本的缓存、prefill 和排队分项。
    ranking = sorted(candidates, key=lambda row: (row["ttft_ms"], -row["cached_tokens"], row["replica"]))  # 按预测 TTFT、命中量和副本ID稳定排序。
    return ranking[0], ranking  # 返回最低成本候选和完整解释排名。
first_choice, first_ranking = prefix_aware_route(requests[0], replicas)  # 对法律请求展示所有候选的中间成本。
print("route-01 候选成本分解")  # 标记下表展示路由器真正比较的中间量。
print("replica      cached  uncached  queue_ms  prefill_ms  total_TTFT")  # 输出候选分解表头。
for row in first_ranking:  # 逐候选展示缓存节省与排队代价。
    print(f"{row['replica']:<12} {row['cached_tokens']:>6} {row['uncached_tokens']:>9} {row['queue_ms']:>9.1f} {row['prefill_ms']:>11.1f} {row['ttft_ms']:>11.1f}")  # 输出当前候选的全部成本项。
print("最终选择=", first_choice)  # 展示成本模型为何为法律请求选择 replica-a。

route-01 候选成本分解
replica      cached  uncached  queue_ms  prefill_ms  total_TTFT
replica-a       920        80      75.0        10.0        85.0
replica-b         0      1000      12.0       125.0       137.0
replica-c         0      1000      48.0       125.0       173.0
最终选择= {'replica': 'replica-a', 'cached_tokens': 920, 'uncached_tokens': 80, 'queue_ms': 75.0, 'prefill_ms': 10.0, 'ttft_ms': 85.0}


## 4. 逐请求结果与结果解读

在同一副本快照和同一批请求上比较两种策略。缓存感知策略不是为了让命中率无限高，而是降低真实 TTFT 并提升 SLO 达标率。

In [4]:
corrected_rows = []  # 保存七条请求的缓存感知路由结果。
for request in requests:  # 逐请求计算最低 TTFT 候选。
    choice, ranking = prefix_aware_route(request, replicas)  # 获取最终选择和完整候选排名。
    corrected_rows.append({"id": request["id"], **choice, "slo_met": choice["ttft_ms"] <= request["slo_ms"], "ranking": ranking})  # 保存路由、SLO 和解释证据。
baseline_average = sum(row["ttft_ms"] for row in baseline_rows) / len(baseline_rows)  # 计算最短队列平均 TTFT。
corrected_average = sum(row["ttft_ms"] for row in corrected_rows) / len(corrected_rows)  # 计算缓存感知平均 TTFT。
baseline_slo_rate = sum(row["slo_met"] for row in baseline_rows) / len(baseline_rows)  # 计算基线 SLO 达标率。
corrected_slo_rate = sum(row["slo_met"] for row in corrected_rows) / len(corrected_rows)  # 计算修正策略 SLO 达标率。
print("请求       baseline(replica/TTFT)       cache-aware(replica/TTFT)    saved_ms  cached")  # 输出逐请求同数据对照表头。
for baseline, corrected in zip(baseline_rows, corrected_rows):  # 逐条比较选择和延迟。
    saved_ms = baseline["ttft_ms"] - corrected["ttft_ms"]  # 计算缓存感知路由节省的毫秒数。
    print(f"{baseline['id']:<10} {baseline['replica']}/{baseline['ttft_ms']:>6.1f}ms      {corrected['replica']}/{corrected['ttft_ms']:>6.1f}ms            {saved_ms:>7.1f} {corrected['cached_tokens']:>7}")  # 输出当前请求的路由收益。
print(f"结果解读：平均TTFT从{baseline_average:.1f}ms降到{corrected_average:.1f}ms，SLO达标率从{baseline_slo_rate:.1%}升到{corrected_slo_rate:.1%}。")  # 解释缓存亲和性对用户延迟的影响。

请求       baseline(replica/TTFT)       cache-aware(replica/TTFT)    saved_ms  cached
route-01   replica-b/ 137.0ms      replica-a/  85.0ms               52.0     920
route-02   replica-b/  18.9ms      replica-b/  18.9ms                0.0     780
route-03   replica-b/ 157.0ms      replica-c/  63.0ms               94.0    1040
route-04   replica-b/ 132.0ms      replica-a/  80.0ms               52.0     920
route-05   replica-b/  32.0ms      replica-b/  32.0ms                0.0     780
route-06   replica-b/ 147.6ms      replica-c/  53.6ms               94.0    1040
route-07   replica-b/ 153.2ms      replica-a/ 101.2ms               52.0     920
结果解读：平均TTFT从111.1ms降到62.0ms，SLO达标率从28.6%升到100.0%。


## 5. 失败案例与修正：只看缓存命中会把请求送进拥塞副本

将 `replica-a` 队列人为提高到 900ms 后，硬亲和性仍选择法律缓存；成本模型会比较“命中但排队”和“miss 但轻载”，转而选择 `replica-b`。

In [5]:
overloaded_replicas = [{**replica, "queue_ms": 900.0 if replica["id"] == "replica-a" else replica["queue_ms"]} for replica in replicas]  # 构造法律缓存副本严重拥塞的故障快照。
affinity_choice = max(overloaded_replicas, key=lambda replica: replica["cache"].get("legal-v3", 0))  # 错误地只按命中 Token 最大值选择副本。
affinity_cost = route_cost(requests[0], affinity_choice)  # 计算硬亲和性造成的真实 TTFT。
overload_choice, overload_ranking = prefix_aware_route(requests[0], overloaded_replicas)  # 用统一成本模型重新路由。
print(f"错误行为：只看缓存选择={affinity_choice['id']}，cached={affinity_cost['cached_tokens']}，TTFT={affinity_cost['ttft_ms']:.1f}ms")  # 展示命中很高但排队更差的反例。
print(f"修正行为：成本模型选择={overload_choice['replica']}，cached={overload_choice['cached_tokens']}，TTFT={overload_choice['ttft_ms']:.1f}ms")  # 展示过载时允许 cache miss 的正确权衡。

错误行为：只看缓存选择=replica-a，cached=920，TTFT=910.0ms
修正行为：成本模型选择=replica-b，cached=0，TTFT=137.0ms


## 6. 生产边界

静态快照没有模拟请求执行后的队列变化。线上需要带时间戳的缓存目录、前缀内容哈希与模型版本、排队预测、热点副本复制、目录陈旧降权、故障域隔离、cache eviction 反馈以及 p95/p99 TTFT 监控。

In [6]:
routing_diagnostics = {"requests": len(requests), "baseline_ttft_ms": baseline_average, "cache_aware_ttft_ms": corrected_average, "baseline_slo_rate": baseline_slo_rate, "cache_aware_slo_rate": corrected_slo_rate, "mean_cached_tokens": sum(row["cached_tokens"] for row in corrected_rows) / len(corrected_rows), "snapshot_is_static": True}  # 汇总路由质量、命中和教学边界。
print("生产监控快照：", routing_diagnostics)  # 输出线上 Prefix Cache 路由器应持续观察的指标。

生产监控快照： {'requests': 7, 'baseline_ttft_ms': 111.10714285714286, 'cache_aware_ttft_ms': 61.964285714285715, 'baseline_slo_rate': 0.2857142857142857, 'cache_aware_slo_rate': 1.0, 'mean_cached_tokens': 914.2857142857143, 'snapshot_is_static': True}


## 7. 最小回归测试

断言只覆盖输入规模、成本分解、整体延迟、SLO 与拥塞反例。

In [7]:
assert len(requests) >= 5 and len(replicas) >= 3  # 保证案例包含足够业务请求和副本候选。
assert all(abs(row["ttft_ms"] - row["queue_ms"] - row["prefill_ms"]) < 1.0e-9 for row in first_ranking)  # 保证路由总成本等于可解释分项之和。
assert corrected_average < baseline_average  # 保证同一批请求的平均 TTFT 确实下降。
assert corrected_slo_rate > baseline_slo_rate  # 保证延迟改善转化为更高 SLO 达标率。
assert affinity_choice["id"] == "replica-a" and overload_choice["replica"] != "replica-a"  # 保证硬亲和性拥塞失败真实复现并被成本模型修正。
assert overload_choice["ttft_ms"] < affinity_cost["ttft_ms"]  # 保证修正选择在反例中具有更低真实 TTFT。